# Phase 1 Integration Test (staging)

Runs the Phase 1 pre-filter service (`pipeline/phase1_filter`) end to end in a
disposable staging environment: creates staging GCP resources, deploys the Cloud
Run service, uploads a small test file set, triggers processing manually, and
verifies the results — including the redelivery/idempotency fix from
[PR #4](https://github.com/marceloamoraes/switchb-plan/pull/4).

**Before running:** authenticate `gcloud` (`gcloud auth login` and
`gcloud auth application-default login`) in the same environment this kernel runs
in, and put a small test corpus (2-3 files per extension, a scanned PDF, a corrupt
file, an oversized file) under `test-data/` next to this notebook.

Run cells top to bottom. Step 14 (teardown) deletes the staging resources — only
run it once you're done.

## 0. Locate the repo root

Works regardless of where Jupyter's working directory ends up, as long as it's inside the repo.

In [ ]:
import pathlib

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "pipeline").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

assert (REPO_ROOT / "pipeline").exists(), "Could not find the repo root — run Jupyter from inside the repo."
print("Repo root:", REPO_ROOT)


## 1. Set variables

Edit `PROJECT_ID` before running. Everything else uses a `-staging` suffix so it can't collide with a future production deployment.

In [ ]:
import os

os.environ["PROJECT_ID"] = "your-gcp-project"  # <-- edit this
os.environ["REGION"] = "us-central1"
os.environ["STAGING_BUCKET"] = os.environ["PROJECT_ID"] + "-switchgear-staging"
os.environ["PHASE1_TOPIC"] = "phase1-trigger-staging"
os.environ["PHASE2_TOPIC"] = "phase2-trigger-staging"

for k in ["PROJECT_ID", "REGION", "STAGING_BUCKET", "PHASE1_TOPIC", "PHASE2_TOPIC"]:
    print(f"{k}={os.environ[k]}")


## 2. Create staging bucket + topics

No GCS notification is created — triggering stays manual, matching `scripts/trigger_phase1.py`.

Also creates a plain pull subscription on the Phase 2 topic purely for verification: Phase 2 isn't deployed for this test, and a topic with zero subscriptions silently drops every message published to it.

In [ ]:
%%bash
gcloud storage buckets create "gs://$STAGING_BUCKET" --location="$REGION"
gcloud pubsub topics create "$PHASE1_TOPIC"
gcloud pubsub topics create "$PHASE2_TOPIC"
gcloud pubsub subscriptions create phase2-verify-sub --topic="$PHASE2_TOPIC"


## 3. Create a dedicated runtime service account (least privilege)

In [ ]:
%%bash
gcloud iam service-accounts create phase1-filter-sa --display-name="Phase 1 filter runtime SA"

gcloud storage buckets add-iam-policy-binding "gs://$STAGING_BUCKET" \
  --member="serviceAccount:phase1-filter-sa@${PROJECT_ID}.iam.gserviceaccount.com" \
  --role="roles/storage.objectAdmin"

gcloud pubsub topics add-iam-policy-binding "$PHASE2_TOPIC" \
  --member="serviceAccount:phase1-filter-sa@${PROJECT_ID}.iam.gserviceaccount.com" \
  --role="roles/pubsub.publisher"


## 4. Build and deploy Phase 1 to staging

Generous timeout/memory to start — LibreOffice conversion and OCR are the slow, memory-hungry paths. Tune down after watching real usage in step 10.

In [ ]:
%%bash
gcloud builds submit "$REPO_ROOT/pipeline/phase1_filter" --tag "gcr.io/$PROJECT_ID/phase1-filter-staging"

gcloud run deploy phase1-filter-staging \
  --image "gcr.io/$PROJECT_ID/phase1-filter-staging" \
  --region "$REGION" --no-allow-unauthenticated \
  --service-account="phase1-filter-sa@${PROJECT_ID}.iam.gserviceaccount.com" \
  --timeout=540 --memory=2Gi --cpu=2 \
  --set-env-vars "PROJECT_ID=$PROJECT_ID,BUCKET_NAME=$STAGING_BUCKET,PHASE2_TOPIC=$PHASE2_TOPIC"


In [ ]:
# %%bash cells don't see Python variables directly, so export REPO_ROOT into the
# environment for the cell above to pick up.
os.environ["REPO_ROOT"] = str(REPO_ROOT)


## 5. Wire up the push subscription

In [ ]:
%%bash
gcloud iam service-accounts create run-invoker --display-name="Pub/Sub push invoker"

gcloud run services add-iam-policy-binding phase1-filter-staging \
  --region="$REGION" \
  --member="serviceAccount:run-invoker@${PROJECT_ID}.iam.gserviceaccount.com" \
  --role="roles/run.invoker"

PHASE1_URL=$(gcloud run services describe phase1-filter-staging --region="$REGION" --format='value(status.url)')

gcloud pubsub subscriptions create phase1-sub-staging \
  --topic="$PHASE1_TOPIC" --push-endpoint="$PHASE1_URL" \
  --push-auth-service-account="run-invoker@${PROJECT_ID}.iam.gserviceaccount.com"


## 6. Sanity-check the local test set

Put 2-3 files per extension (`.doc`, `.docx`, `.pdf`, `.xls`, `.xlsx`, `.xlsm`) under `test-data/` next to this notebook, mixing keyword matches and non-matches, plus:
- at least one **scanned PDF** with no text layer (exercises the OCR fallback)
- one **corrupt file** (e.g. a `.txt` renamed to `.pdf`)
- one **oversized file** (large multi-hundred-page PDF or big spreadsheet)

This cell runs the same parsing + regex logic Phase 1 uses in Cloud Run, locally, before spending any cloud time on it.

In [ ]:
import sys

sys.path.insert(0, str(REPO_ROOT / "pipeline" / "phase1_filter"))
from matcher import is_match
from parsers import extract_text

TEST_DATA_DIR = pathlib.Path("test-data")
test_files = sorted(p for p in TEST_DATA_DIR.glob("*") if p.is_file())
assert test_files, f"No files found in {TEST_DATA_DIR.resolve()} — add your test corpus first."

for path in test_files:
    try:
        text = extract_text(str(path))
        print(f"{path.name}: {'MATCH' if is_match(text) else 'no match'} ({len(text)} chars)")
    except Exception as e:
        print(f"{path.name}: ERROR - {e}")


## 7. Upload the test set

In [ ]:
%%bash
gsutil -m cp -r test-data/* "gs://$STAGING_BUCKET/"


## 8. Dry-run the trigger

Confirms the object count/list looks right before actually publishing anything.

In [ ]:
%%bash
python "$REPO_ROOT/scripts/trigger_phase1.py" --project "$PROJECT_ID" --bucket "$STAGING_BUCKET" --topic "$PHASE1_TOPIC" --dry-run


## 9. Run the trigger for real

In [ ]:
%%bash
python "$REPO_ROOT/scripts/trigger_phase1.py" --project "$PROJECT_ID" --bucket "$STAGING_BUCKET" --topic "$PHASE1_TOPIC"


## 10. Check the logs

A bounded, non-blocking read (rather than `logs tail`, which would block the kernel indefinitely) — re-run this cell to refresh.

In [ ]:
%%bash
gcloud logging read \
  'resource.type="cloud_run_revision" AND resource.labels.service_name="phase1-filter-staging"' \
  --project="$PROJECT_ID" --limit=50 --freshness=10m \
  --format="value(timestamp, textPayload)"


## 11. Verify routing

Archive count should equal your non-matching files; extracted count and pulled Pub/Sub messages should equal your matching files.

In [ ]:
%%bash
echo "--- archive/ ---"
gsutil ls "gs://$STAGING_BUCKET/archive/**" 2>/dev/null
echo "--- extracted/ ---"
gsutil ls "gs://$STAGING_BUCKET/extracted/**" 2>/dev/null
echo "--- phase2-verify-sub messages ---"
gcloud pubsub subscriptions pull phase2-verify-sub --auto-ack --limit=50


## 12. Verify the idempotency fix ([PR #4](https://github.com/marceloamoraes/switchb-plan/pull/4))

Manually redeliver a message for an object you know was already archived, and confirm it's handled gracefully instead of erroring. Edit `ALREADY_ARCHIVED_OBJECT` below to a real path from step 11's `archive/` listing (without the `archive/` prefix).

In [ ]:
os.environ["ALREADY_ARCHIVED_OBJECT"] = "path/to/an/already-archived-file.pdf"  # <-- edit this


In [ ]:
%%bash
gcloud pubsub topics publish "$PHASE1_TOPIC" \
  --attribute=eventType=OBJECT_FINALIZE,bucketId=$STAGING_BUCKET,objectId=$ALREADY_ARCHIVED_OBJECT


Re-run step 10's log cell and confirm it shows `"Already processed, skipping redelivery"` rather than an exception/500.

## 13. Corrupt-file and oversized-file checks

**Corrupt file:** step 10's logs should show the caught exception and a 500 response. Pub/Sub will retry it — expected, since the dead-letter-queue gap (plan step 4b) isn't implemented yet. Clear the stuck retries once confirmed:

In [ ]:
%%bash
python3 -c "import datetime,sys; sys.stdout.write(datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'))" > /tmp/_now_ts
NOW_TS=$(cat /tmp/_now_ts)
gcloud pubsub subscriptions seek phase1-sub-staging --time="$NOW_TS"


**Oversized file:** watch step 10's logs for a timeout (504) or an OOM kill. If either happens, bump `--timeout`/`--memory` in step 4 and redeploy, then re-trigger just that one object:

In [ ]:
os.environ["OVERSIZED_OBJECT_PATH"] = "path/to/oversized-file.pdf"  # <-- edit this


In [ ]:
%%bash
python "$REPO_ROOT/scripts/trigger_phase1.py" --project "$PROJECT_ID" --bucket "$STAGING_BUCKET" --topic "$PHASE1_TOPIC" --prefix "$OVERSIZED_OBJECT_PATH"


## 14. Teardown

Deletes every staging resource created above. Only run this once you're done testing.

In [ ]:
%%bash
gcloud run services delete phase1-filter-staging --region="$REGION" -q
gcloud pubsub subscriptions delete phase1-sub-staging phase2-verify-sub -q
gcloud pubsub topics delete "$PHASE1_TOPIC" "$PHASE2_TOPIC" -q
gsutil -m rm -r "gs://$STAGING_BUCKET"
